<h2>Découverte des Transformers</h2>

<div style="gap : 100px; align: center">
    <div style="display: flex; gap: 100px;">
    <div style="margin: 10px">
        <p>
        Dans ce notebook, nous allons implémenter un Transformer from scratch en suivant l'architecture décrite dans
        <a href="https://arxiv.org/abs/1706.03762"><b>Attention Is All You Need</b></a> (Vaswani et al., 2017).
        </p>
        <p>
        Avant le Transformer, le traitement du langage naturel reposait sur des architectures <b>récurrentes</b> (RNN, LSTM, GRU) et des <b>modèles convolutifs</b> (CNN). Ces architectures traitaient les séquences token par token, de manière séquentielle : pour comprendre le mot n°10, il fallait d'abord avoir traité les mots 1 à 9. Cette dépendance séquentielle posait deux problèmes majeurs :
        </p>
        <ul>
            <li><b>Lenteur d'entraînement</b> : impossible de paralléliser le traitement des tokens d'une même séquence, ce qui rendait l'entraînement sur de grands corpus très coûteux.</li>
            <li><b>Perte d'information sur les longues séquences</b> : plus la séquence est longue, plus le modèle "oublie" le début (le fameux problème du <i>vanishing gradient</i>), malgré des mécanismes comme les cellules LSTM conçues pour atténuer ce problème.</li>
        </ul>
        <p>
        Le Transformer abandonne entièrement la récurrence et les convolutions au profit d'un unique mécanisme : l'<b>attention</b> (plus précisément le <i>self-attention</i> ou <i>scaled dot-product attention</i>). Ce mécanisme permet à chaque token de "regarder" directement tous les autres tokens de la séquence en parallèle, sans passer par des états intermédiaires. Les conséquences sont majeures :
        </p>
        <ul>
            <li><b>Parallélisation massive</b> : pendant l'entraînement, tous les tokens d'une séquence sont traités simultanément, ce qui exploite pleinement la puissance des GPU/TPU.</li>
            <li><b>Modélisation des dépendances longues</b> : un token en fin de phrase peut interagir directement avec un token en début de phrase, sans dégradation du signal.</li>
            <li><b>Scalabilité</b> : cette architecture s'est avérée capable de monter en échelle de manière spectaculaire, ouvrant la voie aux grands modèles de langage (GPT, BERT, T5, LLaMA, etc.).</li>
        </ul>
        <p>
        Il est important de noter que le mécanisme d'attention existait déjà avant le Transformer (notamment dans les travaux de Bahdanau et al., 2014, pour la traduction automatique), mais il était utilisé <b>en complément</b> d'architectures récurrentes. L'innovation clé du papier est de montrer qu'on peut construire un modèle performant basé <b>uniquement</b> sur l'attention — d'où le titre "<i>Attention Is All You Need</i>".
        </p>
        <p>
        Note : si la parallélisation accélère considérablement l'entraînement, la génération de texte (inférence) reste <b>séquentielle</b> dans un Transformer auto-régressif — chaque nouveau token dépend des tokens précédemment générés.
        </p>
    </div>
    <div>
        <img src="https://lesdieuxducode.com/images/blog/pauldenoyes@expaceocom/BERT-Transformer-Architecture-Globale.png" width="480">
    </div>
    </div>
    <div>
        <h3>Complexité par couche</h3>
        <table>
            <thead>
                <tr>
                    <th>Type de couche</th>
                    <th>Complexité par couche</th>
                    <th>Opérations séquentielles</th>
                    <th>Longueur max du chemin</th>
                </tr>
            </thead>
            <tbody>
                <tr>
                    <td>Self-Attention</td>
                    <td>O(n² · d)</td>
                    <td>O(1)</td>
                    <td>O(1)</td>
                </tr>
                <tr>
                    <td>Récurrent (RNN)</td>
                    <td>O(n · d²)</td>
                    <td>O(n)</td>
                    <td>O(n)</td>
                </tr>
                <tr>
                    <td>Convolutif (CNN)</td>
                    <td>O(k · n · d²)</td>
                    <td>O(1)</td>
                    <td>O(log<sub>k</sub>(n))</td>
                </tr>
                <tr>
                    <td>Self-Attention (restreint)</td>
                    <td>O(r · n · d)</td>
                    <td>O(1)</td>
                    <td>O(n/r)</td>
                </tr>
            </tbody>
        </table>
        <p style="font-size: 0.9em; color: gray;">
            <i>n</i> = longueur de la séquence, <i>d</i> = dimension du modèle, <i>k</i> = taille du noyau convolutif, <i>r</i> = taille de la fenêtre d'attention restreinte.
        </p>
    </div>
</div>

In [4]:
import torch
import torch.nn as nn
import math

### 1. Implémentation du Transformer from scratch

On implémente notre propre classe `Transformer` en utilisant uniquement les briques de base de PyTorch (`nn.MultiheadAttention`, `nn.Linear`, `nn.LayerNorm`, `nn.Dropout`), sans recourir aux couches pré-faites `nn.TransformerEncoderLayer` / `nn.TransformerDecoderLayer`.

L'architecture se compose de :
- **Embedding** : transforme chaque token (entier) en un vecteur dense de dimension `d_model`
- **Positional Encoding** : injecte l'information de position via des sinusoïdes (le Transformer n'a pas de notion d'ordre sinon)
- **Encoder** : N couches de [self-attention → LayerNorm → FFN → LayerNorm]
- **Decoder** : N couches de [masked self-attention → LayerNorm → cross-attention avec l'encoder → LayerNorm → FFN → LayerNorm]
- **Prediction Head** : projection linéaire `d_model → vocab_size` + softmax pour obtenir les probabilités du prochain token

In [5]:
# Implémentation du Transformer sans nn.TransformerEncoderLayer et nn.TransformerDecoderLayer pour mieux comprendre les étapes internes
# On se base uniquement sur les composants de base : nn.MultiheadAttention, nn.Linear, nn.LayerNorm, nn.Dropout
# En suivant la structure décrite dans le papier original "Attention is All You Need" de Vaswani et al. (2017)

class Transformer(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, nhead: int, num_layers: int, max_seq_len: int = 512, dropout: float = 0.1):
        super().__init__()
        
        # On définit les composants de l'architectures du Transformer de manière explicite pour mieux comprendre les étapes internes
        self.num_layers = num_layers
        # Couche d'embedding pour les tokens
        self.embedding = nn.Embedding(vocab_size, d_model) 

        # Sub-layers de self attention
        self.cross_attention_layer = nn.MultiheadAttention(d_model, nhead, dropout=dropout)

        # Sub-layers de feed-forward
        self.ffn_layer = nn.Sequential(
            nn.Linear(d_model, d_model * 4), # couche linéaire pour augmenter la dimension
            nn.ReLU(), # activation non linéaire
            nn.Linear(d_model * 4, d_model) # couche linéaire pour revenir à la dimension d_model
        )

        # Couche de normalisation
        self.norm_layer = nn.LayerNorm(d_model)

        # Couche de dropout pour l'entraînement
        self.dropout = nn.Dropout(dropout)

        # Couche linéaire pour projeter les représentations décodées vers les logits sur le vocabulaire
        self.output_projection = nn.Linear(d_model, vocab_size)


    def positional_encoding(self, seq_len: int, d_model: int) -> torch.Tensor:
        """Génère des encodages positionnels pour une séquence de longueur seq_len et une dimension d_model.
        Utilise la formule d'encodage positionnel sinusoïdal du papier original du Transformer.
        Est ajouté aux embeddings pour injecter l'information de position dans les représentations des tokens.
        """
        pe = torch.zeros(seq_len, d_model) # (seq_len, d_model) matrice vide
        position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1) # (seq_len, 1) vecteur de positions
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term) # indices pairs
        pe[:, 1::2] = torch.cos(position * div_term) # indices impairs
        return pe.unsqueeze(0)  # (1, seq_len, d_model)


    def encoder(self, src_token_ids: torch.Tensor) -> torch.Tensor:
        """Encode la séquence d'entrée en utilisant les couches d'attention et de feed-forward définies.
        Prend en entrée les token_ids (batch_size, seq_len, d_model) et retourne les représentations encodées (batch_size, seq_len, d_model).
        """
        # Embedding + Positional Encoding
        x = self.embedding(src_token_ids)  # (batch_size, seq_len, d_model)
        x += self.positional_encoding(x.size(1), x.size(2)).to(x.device)  # Ajout des encodages positionnels
        x = self.dropout(x) # Dropout pour régularisation
        x = x.transpose(0, 1)  # Transformer attend (seq_len, batch_size, d_model)

        for _ in range(self.num_layers):
            # Cross-attention (self-attention dans l'encoder)
            attn_output, _ = self.cross_attention_layer(x, x, x)  # Self-attention (Q=K=V=x, même Query, Key, Value)
            attn_output = self.dropout(attn_output) # Dropout pour régularisation
            x = self.norm_layer(x + attn_output)  # Connexion résiduelle + normalisation

            # Feed-forward
            ffn_output = self.ffn_layer(x)  # Passage dans le feed-forward
            ffn_output = self.dropout(ffn_output) # Dropout pour régularisation
            x = self.norm_layer(x + ffn_output)  # Connexion résiduelle + normalisation

        return x.transpose(0, 1)  # Retour à (batch_size, seq_len, d_model)


    def decoder(self, target_token_ids: torch.Tensor, encoder_output: torch.Tensor) -> torch.Tensor:
        """Décode la séquence cible en utilisant les représentations de l'encoder et les couches d'attention et de feed-forward définies.
        Prend en entrée les target_token_ids (batch_size, seq_len, d_model) et les encoder_output (batch_size, seq_len, d_model) et retourne les représentations décodées (batch_size, seq_len, d_model).
        """
        x = self.embedding(target_token_ids)  # (batch_size, seq_len, d_model)
        x += self.positional_encoding(x.size(1), x.size(2)).to(x.device)  # Ajout des encodages positionnels
        x = self.dropout(x) # Dropout pour régularisation
        x = x.transpose(0, 1)  # (seq_len, batch_size, d_model)
        encoder_output = encoder_output.transpose(0, 1)  # (seq_len, batch_size, d_model)

        for _ in range(self.num_layers):
            # Masked cross-attention
            seq_len = x.size(0)
            mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool().to(x.device) # masque pour empêcher d'attendre les positions futures (matrice diagonale supérieure)
            attn_output, _ = self.cross_attention_layer(x, x, x, key_padding_mask=None, attn_mask=mask)
            attn_output = self.dropout(attn_output) # Dropout pour régularisation
            x = self.norm_layer(x + attn_output)  # Connexion résiduelle + normalisation

            # Cross-attention avec l'output de l'encoder
            attn_output, _ = self.cross_attention_layer(x, encoder_output, encoder_output)
            attn_output = self.dropout(attn_output) # Dropout pour régularisation
            x = self.norm_layer(x + attn_output) # Connexion résiduelle + normalisation

            # Feed-forward
            ffn_output = self.ffn_layer(x)  # Passage dans le feed-forward
            ffn_output = self.dropout(ffn_output) # Dropout pour régularisation
            x = self.norm_layer(x + ffn_output)  # Connexion résiduelle + normalisation

        return x.transpose(0, 1)  # Retour à (batch_size, seq_len, d_model)


    def prediction_head(self, decoder_output: torch.Tensor) -> torch.Tensor:
        """Projette les représentations décodées vers les logits sur le vocabulaire pour faire la prédiction du token suivant.
        Puis utilise softmax pour obtenir des probabilités sur le vocabulaire.
        On va retourner les logits bruts sans appliquer softmax, car la fonction de perte (CrossEntropyLoss) attend des logits et applique elle-même le log-softmax de manière plus stable numériquement.
        Prend en entrée les decoder_output (batch_size, seq_len, d_model) et retourne les probabilités (batch_size, seq_len, vocab_size).
        """
        logits = self.output_projection(decoder_output) # (batch_size, seq_len, vocab_size)
        return logits #torch.softmax(logits, dim=-1) # (batch_size, seq_len, vocab_size)


    def forward(self, src_token_ids: torch.Tensor, tgt_token_ids: torch.Tensor) -> torch.Tensor:
        """Effectue une passe avant complète du Transformer en encodant la séquence source et en décodant la séquence cible.
        Prend en entrée les src_token_ids (batch_size, seq_len) et tgt_token_ids (batch_size, seq_len) et retourne les probabilités (batch_size, seq_len, vocab_size) de la séquence suivante.
        """

        # Pass through encoder and decoder
        encoder_output = self.encoder(src_token_ids)
        decoder_output = self.decoder(tgt_token_ids, encoder_output)

        return self.prediction_head(decoder_output)


### 2. Préparation des données : vocabulaire et tokenisation

Pour tester notre Transformer sur une tâche de traduction français → anglais, on a besoin de :
1. **Un corpus parallèle** : des paires de phrases (source FR, cible EN)
2. **Un vocabulaire partagé** : un dictionnaire unique qui associe chaque mot (des deux langues) à un identifiant entier. On utilise un vocabulaire partagé car le français et l'anglais utilisent le même alphabet et partagent des racines communes.
3. **Des tokens spéciaux** :
   - `<PAD>` (index 0) : remplissage pour uniformiser la longueur des séquences dans un batch
   - `<SOS>` (index 1) : début de séquence, signal au decoder pour commencer la génération
   - `<EOS>` (index 2) : fin de séquence, signal au decoder pour arrêter la génération

In [6]:
# Création d'un vocabulaire très simple 
special_tokens = ["<PAD>", "<SOS>", "<EOS>"] # tokens spéciaux pour le padding, le début et la fin de séquence
corpus = [
    "le chat est sur le tapis",
    "le chien dort dans le jardin",
    "un oiseau chante sur la branche",
    "the cat is on the mat",
    "the dog is sleeping in the garden",
    "the bird is singing on the branch",
]

vocab_idx_to_token = {idx: token for idx, token in enumerate(special_tokens + sorted(set(" ".join(corpus).split())))} # vocabulaire combiné pour les deux langues

vocab = {token: idx for idx, token in enumerate(special_tokens + sorted(set(" ".join(corpus).split())))} # vocabulaire combiné pour les deux langues
vocab_size = len(vocab)

print(f"Vocabulaire combiné ({vocab_size} tokens):")
for token, idx in vocab.items():
    print(f"{token}: {idx}")

Vocabulaire combiné (29 tokens):
<PAD>: 0
<SOS>: 1
<EOS>: 2
bird: 3
branch: 4
branche: 5
cat: 6
chante: 7
chat: 8
chien: 9
dans: 10
dog: 11
dort: 12
est: 13
garden: 14
in: 15
is: 16
jardin: 17
la: 18
le: 19
mat: 20
oiseau: 21
on: 22
singing: 23
sleeping: 24
sur: 25
tapis: 26
the: 27
un: 28


La **tokenisation** convertit chaque phrase en une liste d'entiers (token IDs). On ajoute `<SOS>` au début et `<EOS>` à la fin, puis on complète avec du `<PAD>` pour que toutes les séquences aient la même longueur.

Pour l'entraînement en traduction, on décale la cible :
- **Entrée du decoder** : `<SOS> the cat is on the mat <EOS> <PAD>` → sans le dernier token (`[:, :-1]`)
- **Labels (ce qu'on veut prédire)** : `the cat is on the mat <EOS> <PAD>` → sans le premier token (`[:, 1:]`)

In [8]:
# Tokenisation des phrases d'exemple pour tester le Transformer
src_sentence = corpus[:3] # Les 3 premières phrases comme source
tgt_sentence = corpus[3:] # Les 3 dernières phrases comme cible

# Il faut une max_len commune pour le padding des séquences source et cible
max_len = max(max(len(sentence.split()) for sentence in src_sentence), max(len(sentence.split()) for sentence in tgt_sentence)) + 2 # +2 pour les tokens <SOS> et <EOS>

def tokenize(sentences, vocab, max_len=None):
    token_ids = []
    for sentence in sentences:
        ids = [vocab["<SOS>"]] + [vocab[token] for token in sentence.split()] + [vocab["<EOS>"]]
        token_ids.append(ids)
    
    # Padding à la longueur max du batch
    if max_len is None:
        max_len = max(len(ids) for ids in token_ids)
    
    for i in range(len(token_ids)):
        token_ids[i] = token_ids[i][:max_len] + [vocab["<PAD>"]] * max(0, max_len - len(token_ids[i]))
    
    return torch.tensor(token_ids)

src_token_ids = tokenize(src_sentence, vocab, max_len=max_len) # (batch_size, seq_len)
tgt_token_ids = tokenize(tgt_sentence, vocab, max_len=max_len) # (batch_size, seq_len)

print(f"Token IDs source (batch_size, seq_len): {src_token_ids.shape}\n{src_token_ids}")
print(f"Token IDs cible (batch_size, seq_len): {tgt_token_ids.shape}\n{tgt_token_ids}")


Token IDs source (batch_size, seq_len): torch.Size([3, 9])
tensor([[ 1, 19,  8, 13, 25, 19, 26,  2,  0],
        [ 1, 19,  9, 12, 10, 19, 17,  2,  0],
        [ 1, 28, 21,  7, 25, 18,  5,  2,  0]])
Token IDs cible (batch_size, seq_len): torch.Size([3, 9])
tensor([[ 1, 27,  6, 16, 22, 27, 20,  2,  0],
        [ 1, 27, 11, 16, 24, 15, 27, 14,  2],
        [ 1, 27,  3, 16, 23, 22, 27,  4,  2]])


### 3. Test du modèle (non entraîné)

On instancie le Transformer et on fait une passe avant pour vérifier que les dimensions sont correctes. Le modèle n'est pas encore entraîné, donc les prédictions sont aléatoires — c'est normal. L'important ici c'est de vérifier que tout s'emboîte : `(batch_size, seq_len)` en entrée → `(batch_size, seq_len, vocab_size)` en sortie.

In [9]:
# Passons une phrase d'exemple à travers le modèle pour vérifier que les dimensions sont correctes et que le modèle fonctionne sans erreur
model = Transformer(vocab_size=vocab_size, d_model=64, nhead=4, num_layers=2, max_seq_len=max_len)
output_logits = model(src_token_ids, tgt_token_ids) # (batch_size, seq_len, vocab_size)
output_probs = nn.Softmax(dim=-1)(output_logits)

print(f"Shape des probabilités de sortie (batch_size, seq_len, vocab_size): {output_probs.shape}")
print(f"Exemple de probabilités pour la première phrase cible: {output_probs[0]}")
print(f"Ce qui donne la phrase suivante prédite (en prenant le token avec la probabilité maximale à chaque position):")
predicted_tokens = output_probs.argmax(dim=-1)
for i in range(predicted_tokens.shape[0]):
    predicted_sentence = [vocab_idx_to_token[token] for token in predicted_tokens[i].tolist() if token != vocab["<PAD>"]]
    print(f"Phrase prédite {i+1}: {' '.join(predicted_sentence)}")

# Il dit n'importe quoi car il est pas entraîné, mais au moins on voit que les dimensions sont correctes et que le modèle fonctionne sans erreur !

Shape des probabilités de sortie (batch_size, seq_len, vocab_size): torch.Size([3, 9, 29])
Exemple de probabilités pour la première phrase cible: tensor([[0.0768, 0.0368, 0.0155, 0.0338, 0.0173, 0.0571, 0.0104, 0.0395, 0.0080,
         0.0294, 0.0282, 0.0137, 0.0307, 0.0212, 0.0844, 0.0460, 0.0190, 0.0068,
         0.0315, 0.0373, 0.0535, 0.0272, 0.0543, 0.0930, 0.0273, 0.0282, 0.0215,
         0.0146, 0.0367],
        [0.0568, 0.0261, 0.0301, 0.0317, 0.0297, 0.0440, 0.0176, 0.0175, 0.0148,
         0.0139, 0.0359, 0.0233, 0.0290, 0.0331, 0.0247, 0.0490, 0.0411, 0.0164,
         0.0682, 0.0333, 0.0305, 0.0334, 0.0766, 0.1129, 0.0167, 0.0244, 0.0159,
         0.0202, 0.0333],
        [0.0467, 0.0376, 0.0262, 0.0211, 0.0115, 0.0528, 0.0152, 0.0293, 0.0237,
         0.0178, 0.0421, 0.0271, 0.0257, 0.0204, 0.0561, 0.0780, 0.0201, 0.0057,
         0.0629, 0.0162, 0.0363, 0.0456, 0.0648, 0.0723, 0.0232, 0.0417, 0.0074,
         0.0479, 0.0245],
        [0.0431, 0.0610, 0.0447, 0.0203, 0.0342

### 4. Inférence auto-régressive (modèle non entraîné)

En inférence, on ne dispose pas de la séquence cible — c'est précisément ce qu'on veut générer. On procède donc de manière **auto-régressive** :
1. On encode la phrase source une seule fois
2. On initialise la séquence cible avec `<SOS>`
3. On prédit le token suivant, on l'ajoute à la séquence
4. On répète jusqu'à prédire `<EOS>` ou atteindre une longueur maximale

Sans entraînement, les résultats sont incohérents — le modèle prédit essentiellement au hasard.

In [10]:
# Inférence phrase par phrase avec le modèle non entraîné
# Pour l'inférence, on ne peut pas simplement faire une passe avant complète avec des séquences source et cible complètes, car on n'a pas les séquences cibles complètes à l'avance (c'est ce qu'on veut prédire).
# On doit faire une inférence auto-régressive : on encode la phrase source, puis on génère la phrase cible un token à la fois en utilisant le modèle pour prédire le token suivant jusqu'à ce qu'on atteigne <EOS> ou une longueur maximale.

model.eval()
with torch.no_grad():
    for i in range(src_token_ids.size(0)):
        # Encoder une seule phrase (unsqueeze pour garder la dim batch)
        src = src_token_ids[i].unsqueeze(0)  # (1, seq_len)
        encoder_output = model.encoder(src)  # (1, seq_len, d_model)

        # Commencer avec <SOS>
        generated = [vocab["<SOS>"]]

        # Inférence auto-régressive : générer un token à la fois en utilisant le modèle pour prédire le token suivant jusqu'à ce qu'on atteigne <EOS> ou max_len
        for _ in range(max_len):
            tgt = torch.tensor([generated], dtype=torch.long)  # (1, seq_len_actuel)
            decoder_output = model.decoder(tgt, encoder_output)
            output_logits = model.prediction_head(decoder_output)
            output_probs = nn.Softmax(dim=-1)(output_logits)
            next_token = output_probs[0, -1, :].argmax().item()

            generated.append(next_token)
            if next_token == vocab["<EOS>"]:
                break

        # Affichage : couper au premier <EOS>, ignorer <SOS>
        words = []
        for token_id in generated[1:]:  # skip <SOS>
            if token_id == vocab["<EOS>"]:
                break
            words.append(vocab_idx_to_token[token_id])

        print(f"Source  {i+1}: {src_sentence[i]}")
        print(f"Prédit  {i+1}: {' '.join(words)}")
        print(f"Attendu {i+1}: {tgt_sentence[i]}")
        print()

Source  1: le chat est sur le tapis
Prédit  1: singing oiseau singing singing oiseau singing oiseau oiseau dans
Attendu 1: the cat is on the mat

Source  2: le chien dort dans le jardin
Prédit  2: <PAD> branche singing oiseau singing oiseau oiseau oiseau oiseau
Attendu 2: the dog is sleeping in the garden

Source  3: un oiseau chante sur la branche
Prédit  3: <PAD> la singing la singing oiseau un la la
Attendu 3: the bird is singing on the branch



### 5. Entraînement

On entraîne le modèle sur nos 3 paires de phrases avec :

**Teacher forcing** : pendant l'entraînement, on fournit la vraie séquence cible (décalée) au decoder, pas ses propres prédictions. Cela stabilise et accélère l'apprentissage.

**Décalage source/cible** : le decoder reçoit `tgt[:, :-1]` (sans le dernier token) et la loss compare avec `tgt[:, 1:]` (sans `<SOS>`), pour que chaque position apprenne à prédire le token suivant.

#### Cross-Entropy Loss

La **Cross-Entropy Loss** mesure à quel point les prédictions du modèle sont éloignées de la réalité. Le modèle produit des **logits** (scores bruts) pour chaque token du vocabulaire, puis la loss effectue deux opérations en interne :

**1. Softmax** — transforme les logits en probabilités :

$$p_i = \frac{e^{z_i}}{\sum_{j=1}^{V} e^{z_j}}$$

où $z_i$ est le logit du token $i$ et $V$ la taille du vocabulaire. Chaque $p_i \in [0, 1]$ et $\sum p_i = 1$.

**2. Negative Log-Likelihood** — pénalise la probabilité assignée au bon token :

$$\mathcal{L}_t = -\log(p_{y_t})$$

où $y_t$ est l'index du token attendu (le label) à la position $t$.

**Intuition** : si le modèle assigne une probabilité de 0.9 au bon mot → $-\log(0.9) = 0.10$ (loss faible). S'il assigne 0.01 → $-\log(0.01) = 4.60$ (loss élevée). Le modèle est donc poussé à maximiser la probabilité du bon token.

#### Loss totale sur la séquence

La loss est calculée **uniquement sur les tokens cibles** — l'encoder n'a pas de loss directe, il est entraîné indirectement via les gradients qui remontent du decoder. On cumule les NLL de chaque position cible et on prend la moyenne :

$$\mathcal{L} = -\frac{1}{T} \sum_{t=1}^{T} \log(p_{y_t})$$

où $T$ est le nombre de tokens cibles (hors `<PAD>`).

Exemple concret pour la cible `the cat is on the mat <EOS> <PAD>` :

| Position | Token attendu | Proba prédite | Loss |
|----------|--------------|---------------|------|
| 1 | `the` | 0.8 | $-\log(0.8) = 0.22$ |
| 2 | `cat` | 0.6 | $-\log(0.6) = 0.51$ |
| 3 | `is` | 0.3 | $-\log(0.3) = 1.20$ |
| 4 | `on` | 0.7 | $-\log(0.7) = 0.36$ |
| 5 | `the` | 0.9 | $-\log(0.9) = 0.10$ |
| 6 | `mat` | 0.4 | $-\log(0.4) = 0.92$ |
| 7 | `<EOS>` | 0.5 | $-\log(0.5) = 0.69$ |
| 8 | `<PAD>` | — | **ignoré** |

Loss finale = $(0.22 + 0.51 + 1.20 + 0.36 + 0.10 + 0.92 + 0.69) / 7 = 0.57$

**Pourquoi pas de softmax dans le modèle ?** `nn.CrossEntropyLoss` combine `log_softmax + NLLLoss` en interne de manière numériquement stable. Si on appliquait un softmax avant, la loss referait un `log(softmax(...))` → résultats faux et instabilité numérique. Le modèle doit donc retourner des **logits bruts**.

**`ignore_index=vocab["<PAD>"]`** : les tokens de padding ne participent pas au calcul de la loss — on ne veut pas que le modèle apprenne à prédire du padding.

Avec seulement 3 phrases, le modèle peut les mémoriser parfaitement — ce n'est pas de la généralisation, mais ça valide que l'architecture fonctionne.

In [11]:
# Entrainons le modèle sur les données d'exemple pour voir s'il peut apprendre à faire des prédictions correctes après quelques epochs d'entraînement
model = Transformer(vocab_size=vocab_size, d_model=64, nhead=4, num_layers=2, max_seq_len=max_len)

model.train()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=vocab["<PAD>"]) # Ignorer le token de padding dans le calcul de la perte
num_epochs = 100

device = "mps"
model.to(device)
src_token_ids = src_token_ids.to(device)
tgt_token_ids = tgt_token_ids.to(device)
for epoch in range(num_epochs):
    optimizer.zero_grad()
    output_probs = model(src_token_ids, tgt_token_ids[:, :-1]) # On donne la séquence cible sans le dernier token pour que le modèle puisse prédire le token suivant
    loss = criterion(output_probs.view(-1, vocab_size), tgt_token_ids[:, 1:].reshape(-1)) # On compare avec la séquence cible décalée de 1 (sans <SOS>)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss.item():.4f}")



Epoch 10/100, Loss: 2.0730
Epoch 20/100, Loss: 1.1972
Epoch 30/100, Loss: 0.5971
Epoch 40/100, Loss: 0.4105
Epoch 50/100, Loss: 0.2486
Epoch 60/100, Loss: 0.1692
Epoch 70/100, Loss: 0.1126
Epoch 80/100, Loss: 0.0757
Epoch 90/100, Loss: 0.0573
Epoch 100/100, Loss: 0.0573


### 6. Inférence avec le modèle entraîné

On refait l'inférence auto-régressive, cette fois avec le modèle entraîné. Sur les phrases d'entraînement, le modèle devrait prédire parfaitement (il les a mémorisées).

In [12]:
# Inférence phrase par phrase avec le modèle entraîné
model.to("cpu") # On remet le modèle sur CPU pour l'inférence
src_token_ids = src_token_ids.to("cpu")
model.eval()
with torch.no_grad():
    for i in range(src_token_ids.size(0)):
        # Encoder une seule phrase (unsqueeze pour garder la dim batch)
        src = src_token_ids[i].unsqueeze(0)  # (1, seq_len)
        encoder_output = model.encoder(src)  # (1, seq_len, d_model)

        # Commencer avec <SOS>
        generated = [vocab["<SOS>"]]

        # Inférence auto-régressive : générer un token à la fois en utilisant le modèle pour prédire le token suivant jusqu'à ce qu'on atteigne <EOS> ou max_len
        for _ in range(max_len):
            tgt = torch.tensor([generated], dtype=torch.long)  # (1, seq_len_actuel)
            decoder_output = model.decoder(tgt, encoder_output)
            output_logits = model.prediction_head(decoder_output)
            output_probs = nn.Softmax(dim=-1)(output_logits)
            next_token = output_probs[0, -1, :].argmax().item()

            generated.append(next_token)
            if next_token == vocab["<EOS>"]:
                break

        # Affichage : couper au premier <EOS>, ignorer <SOS>
        words = []
        for token_id in generated[1:]:  # skip <SOS>
            if token_id == vocab["<EOS>"]:
                break
            words.append(vocab_idx_to_token[token_id])

        print(f"Source  {i+1}: {src_sentence[i]}")
        print(f"Prédit  {i+1}: {' '.join(words)}")
        print(f"Attendu {i+1}: {tgt_sentence[i]}")
        print()

Source  1: le chat est sur le tapis
Prédit  1: the cat is on the mat
Attendu 1: the cat is on the mat

Source  2: le chien dort dans le jardin
Prédit  2: the dog is sleeping in the garden
Attendu 2: the dog is sleeping in the garden

Source  3: un oiseau chante sur la branche
Prédit  3: the bird is singing on the branch
Attendu 3: the bird is singing on the branch



### 7. Évaluation sur des phrases inédites

Le vrai test : on donne au modèle des phrases qu'il n'a **jamais vues** pendant l'entraînement. Avec un vocabulaire de 29 tokens et seulement 3 paires d'entraînement, le modèle ne peut pas généraliser — il fait des erreurs sur les combinaisons inédites. C'est attendu : un vrai système de traduction nécessite des millions de paires de phrases.

In [14]:
# Inférence phrase par phrase avec le modèle entraîné sur des phrases d'évaluation différentes de celles d'entraînement
model.to("cpu") # On remet le modèle sur CPU pour l'inférence
model.eval()
src_eval_sentence = [
    "le chat est dans le jardin",
    "un chien dort sur le tapis",
    "un oiseau chante sur la branche"
]

tgt_eval_sentence = [
    "the cat is sleeping in the garden",
    "a dog is sleeping on the mat",
    "a bird is singing on the branch"
]

src_eval_token_ids = tokenize(src_eval_sentence, vocab, max_len=max_len).to("cpu")

model.eval()
with torch.no_grad():
    for i in range(src_eval_token_ids.size(0)):
        # Encoder une seule phrase (unsqueeze pour garder la dim batch)
        src = src_eval_token_ids[i].unsqueeze(0)  # (1, seq_len)
        encoder_output = model.encoder(src)  # (1, seq_len, d_model)

        # Commencer avec <SOS>
        generated = [vocab["<SOS>"]]

        # Inférence auto-régressive : générer un token à la fois en utilisant le modèle pour prédire le token suivant jusqu'à ce qu'on atteigne <EOS> ou max_len
        for _ in range(max_len):
            tgt = torch.tensor([generated], dtype=torch.long)  # (1, seq_len_actuel)
            decoder_output = model.decoder(tgt, encoder_output)
            output_logits = model.prediction_head(decoder_output)
            output_probs = nn.Softmax(dim=-1)(output_logits)
            next_token = output_probs[0, -1, :].argmax().item()

            generated.append(next_token)
            if next_token == vocab["<EOS>"]:
                break

        # Affichage : couper au premier <EOS>, ignorer <SOS>
        words = []
        for token_id in generated[1:]:  # skip <SOS>
            if token_id == vocab["<EOS>"]:
                break
            words.append(vocab_idx_to_token[token_id])

        print(f"Source  {i+1}: {src_eval_sentence[i]}")
        print(f"Prédit  {i+1}: {' '.join(words)}")
        print(f"Attendu {i+1}: {tgt_eval_sentence[i]}")
        print()

Source  1: le chat est dans le jardin
Prédit  1: the cat is on the mat
Attendu 1: the cat is sleeping in the garden

Source  2: un chien dort sur le tapis
Prédit  2: the dog is sleeping in the garden
Attendu 2: a dog is sleeping on the mat

Source  3: un oiseau chante sur la branche
Prédit  3: the bird is singing on the branch
Attendu 3: a bird is singing on the branch



### 8. Comparaison avec un Transformer pré-entraîné

Pour voir ce qu'un Transformer peut réellement faire, on utilise **MarianMT** (`Helsinki-NLP/opus-mt-fr-en`) : une implémentation fidèle de l'architecture originale du papier, entraînée sur des millions de paires de phrases français → anglais (corpus OPUS).


In [15]:
from transformers import MarianMTModel, MarianTokenizer

In [16]:
src_eval_sentence = [
    "le chat est dans le jardin",
    "un chien dort sur le tapis",
    "un oiseau chante sur la branche"
]

tgt_eval_sentence = [
    "the cat is sleeping in the garden",
    "a dog is sleeping on the mat",
    "a bird is singing on the branch"
]

In [17]:
# Traduction avec MarianMT (Helsinki-NLP) pré-entraîné
# MarianMT est une implémentation fidèle du Transformer original, entraînée spécifiquement pour la traduction

model_name = "Helsinki-NLP/opus-mt-fr-en"  # Modèle français → anglais
tokenizer_mt = MarianTokenizer.from_pretrained(model_name)
model_mt = MarianMTModel.from_pretrained(model_name)

model_mt.eval()
for i, src in enumerate(src_eval_sentence):
    # Tokenisation directe (pas de préfixe nécessaire)
    input_ids = tokenizer_mt(src, return_tensors="pt").input_ids

    # Génération auto-régressive
    with torch.no_grad():
        output_ids = model_mt.generate(input_ids, max_length=50)

    predicted = tokenizer_mt.decode(output_ids[0], skip_special_tokens=True)

    print(f"Source  {i+1}: {src}")
    print(f"Prédit  {i+1}: {predicted}")
    print(f"Attendu {i+1}: {tgt_eval_sentence[i]}")
    print()

/opt/miniconda3/envs/myenv/lib/python3.13/site-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

Source  1: le chat est dans le jardin
Prédit  1: the cat is in the garden
Attendu 1: the cat is sleeping in the garden

Source  2: un chien dort sur le tapis
Prédit  2: a dog sleeps on the carpet
Attendu 2: a dog is sleeping on the mat

Source  3: un oiseau chante sur la branche
Prédit  3: a bird sings on the branch
Attendu 3: a bird is singing on the branch

